# Chapter 10 — Build It Yourself: Distributed Training

Data parallelism is just: copy the model onto N workers, give each a batch shard, and **average
their gradients**. Here you'll simulate the all-reduce, **prove** the gradient-averaging is exact,
launch a *real* 4-process DDP run, and compute ZeRO's memory savings.

> 💡 **No GPU needed.** `torch.distributed` runs on the CPU with the `gloo` backend, so everything
> here is real distributed code. (Running a ✍️ cell before you fill it prints a friendly ⚠️, not
> an error — that's expected.)

In [ ]:
import torch
import torch.nn as nn
print('Distributed training runs on the CPU here (gloo). Real GPUs use nccl — same code.')

## Step 1 — all-reduce = averaging  ✍️ Your turn

An **all-reduce (AVG)** takes one tensor from each rank and gives every rank back their average.
That's how DDP combines gradients. Simulate it in one process: average the four ranks' gradients.

<details><summary>Stuck? Show the answer</summary>

```python
averaged = torch.stack(rank_grads).mean(dim=0)
```
</details>

In [ ]:
# four ranks, each holding its own gradient vector
rank_grads = [torch.tensor([1., 2.]), torch.tensor([3., 4.]),
              torch.tensor([5., 6.]), torch.tensor([7., 8.])]

# ✍️ all-reduce AVG: stack the four and take the mean across ranks
averaged = None      # replace
print('averaged gradient ->', averaged)

In [ ]:
# ▶️ Check your work
try:
    if averaged is None:
        print('⚠️  Fill in the ✍️ cell above (stack + mean), then re-run. (Expected until you do.)')
    elif torch.allclose(averaged, torch.tensor([4., 5.])):
        print('✅ all-reduce AVG of [1..8] -> [4, 5], the per-position mean. That is how DDP combines grads.')
    else:
        print(f'⚠️  Got {averaged.tolist()}, expected [4.0, 5.0]. Average ACROSS ranks (dim=0).')
except Exception as e:
    print('❌', e)

## Step 2 — Prove DDP is exact  ✍️ Your turn

The reason averaging shard-gradients is *allowed*: it equals the gradient over the whole batch.
Compute the full-batch gradient, then average the 4 shard gradients, and check they match.

<details><summary>Stuck? Show the answer</summary>

```python
averaged = torch.stack(shard_grads).mean(dim=0)
```
</details>

In [ ]:
def grad_of(model, x, y):
    model.zero_grad(); (((model(x) - y) ** 2).mean()).backward()
    return torch.cat([p.grad.flatten() for p in model.parameters()])

torch.manual_seed(0); model = nn.Linear(4, 2)
X = torch.randn(16, 4); Y = torch.randn(16, 2)
full = grad_of(model, X, Y)                       # single-worker, whole-batch gradient

N = 4
shard_grads = [grad_of(model, X[r::N], Y[r::N]) for r in range(N)]   # each rank's shard grad
# ✍️ average the shard gradients (what DDP's all-reduce does)
averaged = None      # replace
print('max |full - averaged| =', None if averaged is None else (full - averaged).abs().max().item())

In [ ]:
# ▶️ Check your work
try:
    if averaged is None:
        print('⚠️  Fill in the ✍️ cell above (average the shard grads), then re-run. (Expected until you do.)')
    elif torch.allclose(full, averaged, atol=1e-5):
        print('✅ Identical to full-batch (within rounding). Averaging shard-grads == training on the')
        print('   whole batch — DDP changes the speed, not the answer.')
    else:
        print(f'⚠️  Not matching (max diff {(full - averaged).abs().max().item():.2e}). Average across the shards (dim=0).')
except Exception as e:
    print('❌', e)

## Step 3 — A real 4-process DDP run  ▶️ Run this

Now the real thing. `mp.spawn` can't run inside a notebook cell, so we write a small DDP training
script to disk and launch it as a subprocess — 4 real processes, gloo backend. Watch all four
ranks end with the **identical** weight: proof the gradient all-reduce kept them in lockstep.
(This takes a few seconds to spin up the processes.)

In [ ]:
import subprocess, sys, tempfile, os

ddp_script = r'''
import os, torch, torch.nn as nn
import torch.distributed as dist, torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
def worker(rank, world):
    os.environ['MASTER_ADDR'] = '127.0.0.1'; os.environ['MASTER_PORT'] = '29506'
    dist.init_process_group('gloo', rank=rank, world_size=world)
    torch.manual_seed(0); model = nn.Linear(4, 2); ddp = DDP(model)
    opt = torch.optim.SGD(ddp.parameters(), lr=0.1)
    g = torch.Generator().manual_seed(42)
    X = torch.randn(16, 4, generator=g); Y = torch.randn(16, 2, generator=g)
    xb, yb = X[rank::world], Y[rank::world]     # each rank trains on its own shard
    for _ in range(20):
        opt.zero_grad(); (((ddp(xb) - yb) ** 2).mean()).backward(); opt.step()
    print(f'  rank {rank}/{world}: final weight[0,0] = {model.weight[0,0].item():.6f}')
    dist.barrier(); dist.destroy_process_group()
if __name__ == '__main__':
    mp.spawn(worker, args=(4,), nprocs=4, join=True)
    print('All 4 ranks printed the SAME weight -> the gradient all-reduce kept them in sync.')
'''
path = os.path.join(tempfile.gettempdir(), 'ch10_nb_ddp.py')
open(path, 'w').write(ddp_script)
print('launching 4 real processes (gloo)...\n')
try:
    out = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=120)
    print(out.stdout.strip() if out.returncode == 0 else 'unexpected:\n' + out.stderr[-400:])
except subprocess.TimeoutExpired:
    print('(timed out here — run `python ../code/ddp_train.py` in a terminal to see it)')

## Step 4 — ZeRO: shard the memory  ✍️ Your turn

DDP replicates *all* the training state (params + grads + optimizer) on every GPU — redundant.
**ZeRO** shards it: each GPU keeps only 1/N. Fill the three sharding lines and watch the per-GPU
memory of a 7.5B model drop from an impossible 120 GB toward something that fits.

<details><summary>Stuck? Show the answer</summary>

```python
if stage >= 1: optim /= N      # ZeRO-1: shard optimizer state
if stage >= 2: grads /= N      # ZeRO-2: + gradients
if stage >= 3: params /= N     # ZeRO-3 / FSDP: + parameters
```
</details>

In [ ]:
def per_gpu_gb(P, N, stage):
    params, grads, optim = 2, 2, 12    # bytes/param: fp16 p, fp16 g, (fp32 master + Adam m + v)
    # ✍️ shard each piece across N GPUs at the right stage (replace the no-ops)
    if stage >= 1: optim = optim      # replace: optim /= N
    if stage >= 2: grads = grads      # replace: grads /= N
    if stage >= 3: params = params    # replace: params /= N
    return (params + grads + optim) * P / 1e9

P, N = 7.5e9, 8
for stage, name in [(0,'plain DDP'), (1,'ZeRO-1'), (2,'ZeRO-2'), (3,'ZeRO-3/FSDP')]:
    print(f'{name:>12}: {per_gpu_gb(P, N, stage):6.1f} GB / GPU')

In [ ]:
# ▶️ Check your work
try:
    ddp_gb = per_gpu_gb(7.5e9, 8, 0); z3_gb = per_gpu_gb(7.5e9, 8, 3)
    if abs(z3_gb - ddp_gb) < 1:
        print('⚠️  Nothing is sharded yet (ZeRO-3 still equals plain DDP) — fill the /= N lines. (Expected until you do.)')
    elif abs(ddp_gb - 120) < 1 and abs(z3_gb - 15) < 1:
        print(f'✅ Plain DDP {ddp_gb:.0f} GB/GPU (impossible) -> ZeRO-3 {z3_gb:.0f} GB/GPU (fits!). Sharding is how big models train.')
    else:
        print(f'⚠️  Got DDP {ddp_gb:.0f} and ZeRO-3 {z3_gb:.0f} GB; expected 120 and 15. Divide each piece by N at its stage.')
except Exception as e:
    print('❌', e)

## 🎉 You can distribute training now.

You averaged gradients across ranks, proved DDP is exact, ran real 4-process training, and saw
ZeRO shrink a 7.5B model from 120 GB/GPU to 15. Swap `gloo`→`nccl` and each rank onto its own GPU,
and this same code trains at `N×` the speed.

Take it to your **GPT** in the [mini-project](../project/), then on to
[Chapter 11 — Datasets](../../11-datasets/). 🏎️